# Notebook 3 — Data Preprocessing, Applied EDA & Feature Engineering
## Step 3: EDA + Feature Engineering Report

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_dataset
from preprocessor import preprocess, NUMERIC_COLS, CATEGORICAL_COLS
from feature_engineering import (
    add_domain_features, bin_count_features,
    compute_feature_importance, filter_selection, embedded_selection,
    apply_pca, plot_tsne
)

plt.style.use('seaborn-v0_8-whitegrid')

train_df, test_df = load_dataset(data_dir='../data/raw')
print('Loaded:', train_df.shape, test_df.shape)

## 3.1 Data Cleaning

In [ ]:
print('Missing values (train):', train_df.isnull().sum().sum())
print('Duplicate rows  (train):', train_df.duplicated().sum())

# Outlier visualisation for src_bytes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['src_bytes'].plot(kind='box', ax=axes[0], title='src_bytes — before capping')
train_df['src_bytes'].clip(
    upper=train_df['src_bytes'].quantile(0.99)
).plot(kind='box', ax=axes[1], title='src_bytes — after 99th pctile capping')
plt.tight_layout()
plt.savefig('../reports/outlier_capping.png', dpi=150)
plt.show()

## 3.2 Feature Engineering

In [ ]:
# Add domain features before preprocessing
train_fe = add_domain_features(train_df)
test_fe  = add_domain_features(test_df)
train_fe = bin_count_features(train_fe)
test_fe  = bin_count_features(test_fe)

print('New features added:', set(train_fe.columns) - set(train_df.columns))

## 3.3 Applied EDA — Distributions & Correlations

In [ ]:
# Distribution of numeric features by label
key_features = ['src_bytes', 'dst_bytes', 'count', 'serror_rate',
                'byte_ratio', 'error_rate_total', 'log_src_bytes']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, feat in enumerate(key_features[:7]):
    ax = axes[i // 4][i % 4]
    for label_val, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = train_fe[train_fe['binary_label'] == label_val][feat]
        subset_clipped = subset.clip(upper=subset.quantile(0.99))
        subset_clipped.plot(kind='hist', bins=50, ax=ax, alpha=0.6, color=color,
                            label='Normal' if label_val == 0 else 'Attack')
    ax.set_title(feat)
    ax.legend(fontsize=7)

axes[1][3].axis('off')
plt.suptitle('Feature Distributions: Normal vs Attack', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/eda_distributions.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap (top features)
corr_features = ['src_bytes', 'dst_bytes', 'count', 'srv_count', 'serror_rate',
                 'rerror_rate', 'same_srv_rate', 'diff_srv_rate',
                 'dst_host_count', 'dst_host_srv_count', 'binary_label']

plt.figure(figsize=(10, 8))
corr = train_fe[corr_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix — Key Features')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Pairplot for key features (sampled)
sample = train_fe[['log_src_bytes', 'log_dst_bytes', 'error_rate_total',
                   'host_srv_ratio', 'binary_label']].sample(3000, random_state=42)
sample['label_str'] = sample['binary_label'].map({0: 'Normal', 1: 'Attack'})

g = sns.pairplot(sample.drop(columns=['binary_label']),
                 hue='label_str', palette={'Normal': '#2ecc71', 'Attack': '#e74c3c'},
                 plot_kws={'alpha': 0.3, 's': 10})
g.fig.suptitle('Pairplot — Engineered Features', y=1.01, fontsize=12)
plt.savefig('../reports/pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.4 Preprocessing Pipeline

In [ ]:
X_train, X_test, y_train, y_test, feature_names = preprocess(
    train_df, test_df, scaler='standard', save_dir='../models'
)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'Features: {len(feature_names)}')

# Save processed arrays for downstream notebooks
import numpy as np
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_test.npy',  X_test)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy',  y_test)

import json
with open('../data/processed/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('Processed arrays saved.')

## 3.5 Feature Importance

In [ ]:
importance_df = compute_feature_importance(
    X_train, y_train, feature_names, top_n=20, save_dir='../reports'
)
importance_df.head(20)

## 3.6 Feature Selection

In [ ]:
# Filter method (Mutual Information)
X_train_fi, selected_mi, selector_mi = filter_selection(
    X_train, y_train, feature_names, k=20
)
print('MI selected features:', selected_mi)

In [ ]:
# Embedded method (SelectFromModel with GradientBoosting)
X_train_emb, selected_emb, selector_emb = embedded_selection(
    X_train, y_train, feature_names
)
print(f'Embedded selection: {len(selected_emb)} features')
print(selected_emb)

## 3.7 Dimensionality Reduction — PCA

In [ ]:
X_train_pca, X_test_pca, pca = apply_pca(
    X_train, X_test, n_components=0.95, save_dir='../models'
)
print(f'PCA: {X_train.shape[1]} → {X_train_pca.shape[1]} components (95% variance)')

In [ ]:
# t-SNE visualisation (sampled for speed)
plot_tsne(
    X_train_pca, y_train,
    title='t-SNE: NSL-KDD (PCA-reduced, Normal vs Attack)',
    save_path='../reports/tsne_plot.png',
    sample_size=3000
)

from IPython.display import Image
Image('../reports/tsne_plot.png')